# 3.3 · Prompts and argument completion

**提示（Prompts）与参数补全**

> **After this lesson you can:** Publish user-invoked prompt templates with arguments and multi-message, resource-rich results, and add argument autocompletion for prompts and resource templates.

Official reading: [Spec: Prompts](https://modelcontextprotocol.io/specification/2026-07-28/server/prompts) · [Spec: Completion](https://modelcontextprotocol.io/specification/2026-07-28/server/utilities/completion)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A server with three kinds of prompts
`@server.prompt()` turns a function into a prompt: parameters become arguments (required when they have no default, described with `Field(description=...)`), the docstring becomes the description. The function can return a string (one user message) or a list of `UserMessage` / `AssistantMessage` and raw content blocks.

In [ ]:
import json
import logging
from typing import Annotated

from pydantic import Field

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import AssistantMessage, UserMessage
from mcp.shared.exceptions import MCPError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1800):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


STYLE_GUIDE = "1. Type hints everywhere.\n2. No bare except.\n3. Functions under 40 lines."

server = MCPServer("dev-workflows")


@server.resource("docs://style-guide", mime_type="text/markdown")
def style_guide() -> str:
    return STYLE_GUIDE


@server.prompt(title="Request Code Review")
def code_review(code: Annotated[str, Field(description="The code to review")],
                language: Annotated[str, Field(description="Programming language")] = "python") -> str:
    """Asks the LLM to analyze code quality and suggest improvements."""
    return f"Please review this {language} code:\n{code}"


@server.prompt(title="Plan a vacation")
def plan_vacation(destination: Annotated[str, Field(description="City or region")],
                  days: Annotated[int, Field(description="Trip length in days (whole number)")] = 7) -> list:
    """Guide the model through planning a trip, with a worked example of the expected format."""
    return [
        UserMessage("Plan a 2-day trip to Lisbon."),
        AssistantMessage("Day 1: Alfama + tram 28. Day 2: Belém + LX Factory. Budget: ~€180."),
        UserMessage(f"Great format. Now plan a {days}-day trip to {destination}. "
                    "Check my calendar resource and the weather tool before proposing dates."),
    ]


@server.prompt(title="Review against the style guide")
def review_with_guide(code: Annotated[str, Field(description="The code to review")]) -> list:
    """Review code against the team style guide, which is embedded in the conversation."""
    return [
        UserMessage(types.EmbeddedResource(type="resource", resource=types.TextResourceContents(
            uri="docs://style-guide", mime_type="text/markdown", text=STYLE_GUIDE))),
        UserMessage(f"Review this code strictly against the style guide above:\n{code}"),
    ]


print("prompts:", [p.name for p in await server.list_prompts()])

### 2. `prompts/list`: what a slash-command menu needs
Arguments carry `name`, `description` and `required` — but no type. A host renders this as `/code_review <code> [language]`.

In [ ]:
async with Client(server) as client:
    listing = await client.list_prompts()
    print("capabilities.prompts:", client.server_capabilities.prompts)
    show(listing.prompts[0])
    print()
    for p in listing.prompts:
        args = " ".join(f"<{a.name}>" if a.required else f"[{a.name}]" for a in p.arguments or [])
        print(f"/{p.name:<18} {args:<22} {p.title}")

### 3. `prompts/get`: render with arguments
Arguments travel as strings; `MCPServer` converts `"5"` to `int` for `days` using the type hint. Look at the roles in the multi-message prompt and the embedded resource block.

In [ ]:
async with Client(server) as client:
    print("--- code_review")
    show(await client.get_prompt("code_review", {"code": "def hello():\n    print('world')"}))
    print("\n--- plan_vacation (days arrives as the string '5')")
    for m in (await client.get_prompt("plan_vacation", {"destination": "Barcelona", "days": "5"})).messages:
        print(f"{m.role:>9}: {m.content.text}")
    print("\n--- review_with_guide")
    show(await client.get_prompt("review_with_guide", {"code": "try:\n    run()\nexcept:\n    pass"}))

### 4. A host's slash-command handler
The host — not the protocol — turns `PromptMessage`s into its model's message format. Here: parse `/command key=value`, call `prompts/get`, and convert to Anthropic-style messages (embedded resources become tagged text; consecutive same-role messages are merged because many chat APIs require alternation).

In [ ]:
import shlex


def to_chat_messages(prompt_messages):
    chat = []
    for m in prompt_messages:
        c = m.content
        if c.type == "text":
            block = {"type": "text", "text": c.text}
        elif c.type == "resource":
            body = getattr(c.resource, "text", "<binary>")
            block = {"type": "text", "text": f"<resource uri='{c.resource.uri}'>\n{body}\n</resource>"}
        elif c.type == "image":
            block = {"type": "image", "source": {"type": "base64", "media_type": c.mime_type, "data": c.data}}
        else:
            block = {"type": "text", "text": f"[{c.type}]"}
        if chat and chat[-1]["role"] == m.role:
            chat[-1]["content"].append(block)
        else:
            chat.append({"role": m.role, "content": [block]})
    return chat


async def run_slash_command(client, line: str):
    name, *pairs = shlex.split(line.lstrip("/"))
    arguments = dict(pair.split("=", 1) for pair in pairs)
    result = await client.get_prompt(name, arguments)
    return to_chat_messages(result.messages)


async with Client(server) as client:
    messages = await run_slash_command(client, "/review_with_guide code='except: pass'")
print(json.dumps(messages, indent=2))

### 5. Break it: bad arguments and the error code
The spec says unknown prompt names and missing required arguments SHOULD be `-32602`. See what the SDK actually sends.

In [ ]:
async with Client(server) as client:
    for label, name, args in [("missing required arg", "code_review", {}),
                              ("unknown prompt", "deploy", {}),
                              ("non-numeric days", "plan_vacation", {"destination": "Rome", "days": "a week"})]:
        try:
            await client.get_prompt(name, args)
        except MCPError as err:
            print(f"{label:<22} -> code={err.code} message={err.message!r}")

All three come back as `-32603` "Internal server error" — an SDK gap, and unhelpful to a user filling in a form. Fix it in the handler: accept the raw string, validate, and raise `MCPError` with `INVALID_PARAMS`, which the SDK passes through unchanged.

In [ ]:
from mcp_types import INVALID_PARAMS


@server.prompt(title="Plan a vacation (validated)")
def plan_trip(destination: Annotated[str, Field(description="City or region")],
              days: Annotated[str, Field(description="Trip length in days, 1-30")] = "7") -> str:
    """Plan a trip; validates its arguments and reports problems as -32602."""
    if not days.isdigit() or not 1 <= int(days) <= 30:
        raise MCPError(INVALID_PARAMS, f"Argument 'days' must be a whole number from 1 to 30, got {days!r}",
                       data={"argument": "days"})
    return f"Plan a {int(days)}-day trip to {destination}."


async with Client(server) as client:
    try:
        await client.get_prompt("plan_trip", {"destination": "Rome", "days": "a week"})
    except MCPError as err:
        print("fixed:", err.code, err.message, err.data)

### 6. Completion for prompt arguments and template variables
One `@server.completion()` handler serves every `completion/complete` request; it receives the `ref`, the `argument` (`name`, `value`) and the `context` with already-resolved arguments. Registering it makes the server advertise the `completions` capability. We cap values at 100 and report `total` / `hasMore`.

In [ ]:
LANGUAGES = ["python", "pytorch", "pyside", "perl", "php", "javascript", "java", "go", "rust"]
FRAMEWORKS = {"python": ["flask", "fastapi", "django"], "javascript": ["express", "fastify", "next"]}
CITIES = ["Paris", "Park City", "Parma", "Barcelona", "Barbados", "Berlin"]


@server.resource("weather://forecast/{city}/{date}", mime_type="application/json")
def forecast(city: str, date: str) -> str:
    return json.dumps({"city": city, "date": date, "high_c": 27})


@server.prompt()
def scaffold(language: str, framework: str) -> str:
    """Scaffold a new web service."""
    return f"Create a {framework} service in {language}."


def ranked(options, prefix, limit=100):
    matches = [o for o in options if o.lower().startswith(prefix.lower())]
    return types.Completion(values=matches[:limit], total=len(matches), has_more=len(matches) > limit)


@server.completion()
async def complete(ref, argument, context):
    resolved = (context.arguments if context and context.arguments else {})
    if isinstance(ref, types.PromptReference):
        if argument.name == "language":
            return ranked(LANGUAGES, argument.value)
        if argument.name == "framework":
            return ranked(FRAMEWORKS.get(resolved.get("language", ""), []), argument.value)
    if isinstance(ref, types.ResourceTemplateReference) and ref.uri == "weather://forecast/{city}/{date}":
        if argument.name == "city":
            return ranked(CITIES, argument.value)
    return None   # no suggestions -> empty values


prompt_ref = types.PromptReference(type="ref/prompt", name="scaffold")
template_ref = types.ResourceTemplateReference(type="ref/resource", uri="weather://forecast/{city}/{date}")

async with Client(server) as client:
    print("completions capability declared:", client.server_capabilities.completions is not None)
    show(await client.complete(prompt_ref, {"name": "language", "value": "py"}))
    print("framework 'f' with language=python:", (await client.complete(
        prompt_ref, {"name": "framework", "value": "f"}, {"language": "python"})).completion.values)
    print("framework 'f' with language=javascript:", (await client.complete(
        prompt_ref, {"name": "framework", "value": "f"}, {"language": "javascript"})).completion.values)
    print("city 'Par' for the forecast template:", (await client.complete(
        template_ref, {"name": "city", "value": "Par"})).completion.values)

### 7. Completion on the wire, and a server without the capability
The raw `completion/complete` exchange over stdio, then the same request to a server that never registered a completion handler: `-32601` method not found, exactly as the spec lists.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "complete_server.py").write_text(textwrap.dedent('''
    import sys
    import mcp_types as types
    from mcp.server import MCPServer
    server = MCPServer("completer")

    @server.prompt()
    def code_review(code: str, language: str) -> str:
        """Review code."""
        return f"Review this {language} code: {code}"

    if "--with-completion" in sys.argv:
        @server.completion()
        async def complete(ref, argument, context):
            langs = ["python", "pytorch", "pyside", "perl", "php"]
            hits = [l for l in langs if l.startswith(argument.value)]
            return types.Completion(values=hits[:3], total=len(hits), has_more=len(hits) > 3)

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}
request = {"jsonrpc": "2.0", "id": 1, "method": "completion/complete",
           "params": {"ref": {"type": "ref/prompt", "name": "code_review"},
                      "argument": {"name": "language", "value": "p"}, "_meta": META}}

for flags in (["--with-completion"], []):
    proc = subprocess.Popen([sys.executable, str(workdir / "complete_server.py"), *flags], stdin=subprocess.PIPE,
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
    proc.stdin.write(json.dumps(request) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)
    print(f"server flags={flags}\n>>> {json.dumps({k: v for k, v in request['params'].items() if k != '_meta'})}")
    print("<<<", json.dumps(reply), "\n")
    proc.stdin.close()
    proc.wait(timeout=10)

### 8. Debouncing on the client
Clients SHOULD debounce. Simulate a user typing `b`, `ba`, `bar`, `barc` quickly, then pausing: only the last value is sent.

In [ ]:
import asyncio

sent = []


async def debounced_complete(client, ref, arg_name, keystrokes, delay=0.15):
    pending = None

    async def fire(value):
        await asyncio.sleep(delay)
        sent.append(value)
        return (await client.complete(ref, {"name": arg_name, "value": value})).completion.values

    for value, gap in keystrokes:
        if pending:
            pending.cancel()
        pending = asyncio.create_task(fire(value))
        await asyncio.sleep(gap)
    return await pending


async with Client(server) as client:
    typing = [("b", 0.03), ("ba", 0.03), ("bar", 0.03), ("barc", 0.0)]
    values = await debounced_complete(client, template_ref, "city", typing)
print("keystrokes:", [k for k, _ in typing], "| requests actually sent:", sent, "| suggestions:", values)

### Your turn
1. Add a prompt `/summarize_meetings` with a `date` argument and an `audience` argument (`team` or `exec`), returning a user message plus an `AssistantMessage` that sets the format. Add completion for `audience`.
2. Make `review_with_guide` return a `types.ResourceLink` to `docs://style-guide` instead of embedding it. Update `to_chat_messages` so the host *fetches* linked resources with `resources/read` before calling the model. When is each approach better?
3. Change `ranked` so it falls back to substring (fuzzy-ish) matching when no prefix matches, and make the city completion return at most 2 values. Check `total` and `hasMore`.
4. Information disclosure: give the completion handler a list of "private projects" and make it return only those the caller may see. Where in a real server would the caller's identity come from? (Hint: lesson 6.1.)